In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

import pickle
import numpy as np
import matplotlib.pyplot as plt

In [7]:
with open('../../data/cifar-10-batches-py/data_batch_1', 'rb') as f:
    data = pickle.load(f, encoding='bytes')
    
X = data[b'data']
y = data[b'labels']

In [11]:
X[0]

array([ 59,  43,  50, ..., 140,  84,  72], shape=(3072,), dtype=uint8)

In [ ]:
class SimpleUnet(nn.Module):
    def __init__(self, dim_in: int):
        super(SimpleUnet, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(dim_in, 64, kernel_size=3, padding=1),
            nn.GELU(),
            nn.AvgPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.GELU(),
            nn.AvgPool2d(2)
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2),
            nn.GELU(),
            nn.ConvTranspose2d(64, 3, kernel_size=2, stride=2),
            nn.GELU()
        )

    def forward(self, x):
        x = self.decoder(self.encoder(x))
        return x

array([[ 59,  43,  50, ..., 140,  84,  72],
       [154, 126, 105, ..., 139, 142, 144],
       [255, 253, 253, ...,  83,  83,  84],
       ...,
       [ 71,  60,  74, ...,  68,  69,  68],
       [250, 254, 211, ..., 215, 255, 254],
       [ 62,  61,  60, ..., 130, 130, 131]],
      shape=(10000, 3072), dtype=uint8)

In [ ]:
class CIFAR10Batch(data):
    def __init__(self, X, y, transform=None):
        self.X = X.reshape(-1, 3, 32, 32)  # (N, 3, 32, 32)
        self.y = np.array(y)
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        img = self.X[idx]
        label = self.y[idx]

        img = img.astype('uint8')

        if self.transform:
            img = self.transform(img)

        return img, label